### CHECK 001_INITIAL SETUP under _ how to setup folder
FOR PYTHON AND ENVIRONMENT VARIABLES

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## LANG CHAIN FUNDAMENTALS 
### Exploring the world of langchain, agents, models, rag, vector db

- Lets begin with how to spin up agents using langchain

In [19]:
from langchain_ollama import ChatOllama

# we will be using ollama model for this notebook, since we are running out of credits 
llama = "llama3.2:3b"

llamamodel = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)


### 1. How to create an agent with tools
- Agent is simply a abstract entity that handles some kind of query, goes to tools if available and generates a response
- How to do ? 

- Import create agents from langchain.agents, create an agent, add a model and invoke the agent simply

In [ ]:
# imports 
from langchain.agents import create_agent
from langchain.tools import tool
# create an agent

@tool
def get_current_news_of_country(country: str) -> str:
    """ This is documentation or more like description of tool 
        This tool is used to get the current news of a country
    """

    # do anything here, call db, external api, collect information and return as string dump to the agent
    # rest agent will figure out how to use this information to answer the question of user
    return f"Current news of {country} is: new government in action"

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools = [get_current_news_of_country]
)

# invoke 

user_message = {
    "role": "user",
    "content": "What is the current news of Nepal?"
}

# response = agent.invoke({"messages" : [user_message]}) # this will return answer
# response

{'messages': [HumanMessage(content='What is the current news of Nepal?', additional_kwargs={}, response_metadata={}, id='7f6df89e-1db9-44a0-83d3-ef4de0d783bc'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_news_of_country', 'arguments': '{"country": "Nepal"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec244-2fd1-7030-9904-a355ffe9a5bb-0', tool_calls=[{'name': 'get_current_news_of_country', 'args': {'country': 'Nepal'}, 'id': 'aa83dfb3-a176-44e3-838a-183006a63851', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 71, 'output_tokens': 21, 'total_tokens': 92, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='Current news of Nepal is: new government in action', name='get_current_news_of_country', id='312c024e-4281-4ac7-ad64-d5115d2eb847', tool_call_id='aa83dfb3-a176-44e3-838a-183006a63851'),
  A

### 2. How to create a model
 

In [30]:
# imports 
from langchain.chat_models import init_chat_model

# create a chat model
# model = create_chat_model("gle_genai:gemini-2.5-flash-lite")
model = init_chat_model(
    model=llama,
    model_provider="ollama"
)

# invoke
response = model.invoke("What is the weather like?") # this will return answer
response.content # this will print the answer to the user question

# we can extend tool
model_extended = model.bind_tools([get_current_news_of_country])

# invoke and it will also traverse the tools 
response = model_extended.invoke("What is the current news of Nepal?") # this will return answer
response # this will print the answer to the user question


AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-13T18:38:22.12353Z', 'done': True, 'done_reason': 'stop', 'total_duration': 632976291, 'load_duration': 178054666, 'prompt_eval_count': 177, 'prompt_eval_duration': 32683999, 'eval_count': 22, 'eval_duration': 414521000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ec247-1f31-77b3-ac87-3410552a0488-0', tool_calls=[{'name': 'get_current_news_of_country', 'args': {'country': 'Nepal'}, 'id': 'ef11aea0-717e-49be-b543-ed2b0c087df6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 177, 'output_tokens': 22, 'total_tokens': 199})

### 3. Streaming and batch 

In [32]:
user_message_2 = {
    "role": "user",
    "content": "Write a 100 word haiku?"
}

# streaming

for chunk in model.stream(user_message_2["content"]):
    print(chunk.text, end=" | ", flush=True)

# batch

responses = model.batch([
    "Write a 100 word haiku",
    "Write a 100 word poem about the sea",
])

for response in responses:
    print(response.text)

Soft | ly |  falls |  the |  dew | 
 | G | ent | le |  morning | 's |  peaceful |  h | ush | 
 | Nature | 's |  calm |  delight |  |  | Softly falls the night
Stars above, a twinkling sea
Moonlight on my face
The ocean's waves crash on the shore,
A soothing melody evermore.
The tide rises, falls with grace,
A constant rhythm in its place.

The salty scent of seaweed fills the air,
As seagulls soar, without a care.
The sun sets low, a fiery ball,
Dancing on the water's surface tall.

In its depths, a world unseen lies,
Where creatures lurk, with curious eyes.
The sea's vast power, we can't define,
A mystery that's always on our mind.

With every wave, our spirits soar,
As the ocean's beauty we adore.


### 4. Tool Execution Loops

In [39]:
# generate tool calls
messages = [{"role": "user", "content": "What is the current news of Nepal?"}]
ai_response = model_extended.invoke(messages)
messages.append(ai_response)


#execute tools
for tool_call in ai_response.tool_calls:
    tool_response = get_current_news_of_country.invoke(tool_call)
    messages.append(tool_response)

#pass to response 
final_response = model_extended.invoke(messages)
print(final_response.text)

# how does this work 
a = """
    The process works as follows:

1. The user sends a message to the model, asking for the current news of Nepal.

2. The model generates a response, which includes tool calls if it determines that it needs to use a tool to answer the question.

3. The tool calls are executed, and the responses from the tools are collected.

4. The original messages, along with the tool responses, are passed back to the model.

5. The model generates a final response based on the combined information from the user message and the tool responses.

"""

The newly formed government in Nepal has taken office, marking a significant shift in the country's political landscape. The Communist Party of Nepal (Maoist Centre) led by Pushpa Kamal Dahal 'Prachanda' is set to implement its manifesto, which includes key policies such as land reform, public sector privatization, and social welfare programs.

The government has also announced plans to address long-standing issues such as poverty, inequality, and corruption. Additionally, the new administration has expressed commitment to strengthening Nepal's relations with neighboring countries and promoting economic growth.

However, the transition to a new government has not been without challenges. The country is still recovering from the devastating effects of the 2015 earthquake, and many Nepalis are concerned about the impact of policy changes on their livelihoods.

As the government gets underway, it remains to be seen how effectively it will address these concerns and implement its policies 

In [35]:
messages

[{'role': 'user', 'content': 'What is the current news of Nepal?'},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-06-13T18:42:48.281071Z', 'done': True, 'done_reason': 'stop', 'total_duration': 818265125, 'load_duration': 200659000, 'prompt_eval_count': 177, 'prompt_eval_duration': 102652000, 'eval_count': 22, 'eval_duration': 499362000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019ec24b-2e25-7af2-8cbc-c5f3ab7c80ba-0', tool_calls=[{'name': 'get_current_news_of_country', 'args': {'country': 'Nepal'}, 'id': '20b12f29-946d-4fda-b037-b17c3bb299e4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 177, 'output_tokens': 22, 'total_tokens': 199}),
 ToolMessage(content='Current news of Nepal is: new government in action', name='get_current_news_of_country', tool_call_id='20b12f29-946d-4fda-b037-b17c3bb299e4')]

### 5. Messages
There are mainly 3 kinds of messages
- System Message
- AI Message
- System Message
- Tool Message

System Message:
- How the model should behave
- More like instruction

AI Message:
- Message generated by LLM as output

Human Message:
- Message from User

Tool message:
- What the model is asking the tool to do

Usage:
```
from langchain.messages import HumanMessage, AIMessage, SystemMessage

messages = [
    SystemMessage("You are a poet."),
    HumanMessage("Write a 20 words poem about the moon."),
]
````

In [36]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# message format 
message_prompt_format = {
    "role": "user",
    "content": "What is AI?",
    "metadata": {
        "source": "user_input",
        "user_name": "Tilak"
    }
}


messages = [
    SystemMessage("You are a poet."),
    HumanMessage("Write a 20 words poem about the moon."),
    AIMessage("The moon is bright, shining in the night, guiding us with its gentle light.")
]

# pass messages to model or agent 
response = model.invoke(messages)
response.content # <- response is the AI message basically


' Its beauty is pure delight.'

Example 2: message in depth in real use

In [38]:
from langchain.messages import SystemMessage, HumanMessage

system_message = SystemMessage(
    """
        You are a senior python developer with 10 years of experience. 
        You are an expert in writing clean and efficient code. 
        You have a deep understanding of python libraries and frameworks. 
        You are also a great teacher and can explain complex concepts in a simple way.
    """
)

messages = [
    system_message,
    HumanMessage("How to create rest api."),
]

response = model.invoke(messages)
response.content

'Creating a REST API with Python is a straightforward process that involves several steps. Here\'s a step-by-step guide on how to create a basic REST API using Python:\n\n**Step 1: Choose a Framework**\n\nThere are several frameworks available for creating REST APIs in Python, including:\n\n* Flask: A lightweight and flexible framework ideal for small projects.\n* Django: A high-level framework that provides an out-of-the-box solution for building complex web applications.\n* FastAPI: A modern framework that provides high-performance and automatic API documentation.\n\nFor this example, we\'ll use Flask, which is a popular choice among Python developers.\n\n**Step 2: Set up the Project**\n\nCreate a new directory for your project and create a `main.py` file inside it:\n```bash\nmkdir my_rest_api\ncd my_rest_api\ntouch main.py\n```\n**Step 3: Install Dependencies**\n\nInstall Flask using pip:\n```bash\npip install flask\n```\n**Step 4: Define Routes**\n\nIn `main.py`, define the routes 